PART A – KNN CLASSIFICATION 

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)
n = 1000
age = np.random.randint(18, 70, n)
monthly_bill = np.random.randint(200, 2000, n)
support_calls = np.random.randint(0, 10, n)
tenure = np.random.randint(1, 72, n)
churn_score = (
0.04 * support_calls
+ 0.002 * monthly_bill
- 0.03 * tenure
+ 0.01 * age
)


churn_probability = 1 / (1 + np.exp(-(
churn_score - 3
)))
churn = np.random.binomial(
1,
churn_probability
)
df = pd.DataFrame({
"Age": age,
"Monthly_Bill": monthly_bill,
"Support_Calls": support_calls,
"Tenure": tenure,
"Churn": churn
})
print(df.head())
print(df.shape)
print(df["Churn"].value_counts())

   Age  Monthly_Bill  Support_Calls  Tenure  Churn
0   56           938              2      24      0
1   69          1238              9      36      1
2   46          1415              6      30      0
3   32          1854              9      24      1
4   60           684              1      30      0
(1000, 5)
Churn
0    726
1    274
Name: count, dtype: int64


In [2]:
df

,Age,Monthly_Bill,Support_Calls,Tenure,Churn
0,56,938,2,24,0
1,69,1238,9,36,1
2,46,1415,6,30,0
3,32,1854,9,24,1
4,60,684,1,30,0
...,...,...,...,...,...
995,60,1230,5,41,0
996,64,228,8,19,1
997,62,1106,5,46,0
998,35,1256,8,28,0


This is a synthetic dataset. It creates a simplified relationship between customer characteristics and churn, not a real-world customer behavior model.

 Separate Features and Target

In [2]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X contains independent variables.
y contains the target variable.

 Train-Test Split

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42,
stratify=y
)

We use 80% for training and 20% for testing.
stratify=y helps preserve the class distribution.

 Apply Feature Scaling

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Why do we fit only on training data?
Because test data represents unseen data.
Using test data to calculate scaling parameters would introduce data leakage.

Build the KNN Classification Model

In [10]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(
n_neighbors=5,
weights="uniform", # uniform weight to each data point
metric="minkowski",
p=2
)

Parameter explanation:
 n_neighbors=5: Considers five nearest observations.
 weights=“uniform”: Gives equal weight to every neighbor.
 metric=“minkowski”: Uses Minkowski distance.
 p=2: Makes Minkowski distance equivalent to Euclidean distance.

12. Train the Model

In [11]:
knn_model.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int32](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


KNN is called a lazy learning algorithm.
Unlike many other algorithms, it does not learn a compact set of model coefficients during training.
It stores the training observations and performs neighbor searches when predictions are requested.

13. Make Predictions

In [13]:
y_pred = knn_model.predict(X_test_scaled)
y_prob = knn_model.predict_proba(X_test_scaled)[:, 1]

 y_pred: Predicted class labels.
 y_prob: Estimated probability of class 1.

14. Evaluate the Classification Model

In [14]:
from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
confusion_matrix,
classification_report,
roc_auc_score
)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(
y_test, y_pred, zero_division=0
))
print("Recall:", recall_score(
y_test, y_pred, zero_division=0
))
print("F1 Score:", f1_score(
y_test, y_pred, zero_division=0
))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(
y_test, y_pred, zero_division=0
))

Accuracy: 0.755
Precision: 0.59375
Recall: 0.34545454545454546
F1 Score: 0.4367816091954023
ROC AUC: 0.726771159874608
Confusion Matrix:
[[132  13]
 [ 36  19]]
Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.91      0.84       145
           1       0.59      0.35      0.44        55

    accuracy                           0.76       200
   macro avg       0.69      0.63      0.64       200
weighted avg       0.73      0.76      0.73       200



Why these metrics?
Metric Meaning
Accuracy Overall proportion of correct predictions
Precision Of predicted positive observations, how many were actually positive?
Recall Of actual positive observations, how many were identified?
F1 Score Harmonic mean of precision and recall
ROC-AUC Measures ranking ability across classification thresholds

For churn or fraud detection, accuracy alone may not be sufficient, especially when classes are imbalanced.

15. How to Find the Best K?
We should not blindly select K = 5.
We can test different K values.

In [9]:
from sklearn.metrics import accuracy_score
k_values = range(1, 21)
results = []
for k in k_values:
    model = KNeighborsClassifier(
    n_neighbors=k
    )
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    accuracy = accuracy_score(
    y_test,
    predictions
    )
    results.append({
    "K": k,
    "Accuracy": accuracy
    })
results_df = pd.DataFrame(results)
print(results_df)

     K  Accuracy
0    1     0.710
1    2     0.740
2    3     0.740
3    4     0.755
4    5     0.755
5    6     0.755
6    7     0.750
7    8     0.735
8    9     0.745
9   10     0.760
10  11     0.750
11  12     0.755
12  13     0.750
13  14     0.755
14  15     0.760
15  16     0.755
16  17     0.765
17  18     0.765
18  19     0.770
19  20     0.765


This gives us accuracy for different K values.
However, selecting K using the final test set repeatedly can lead to test-set overfitting.
A better approach is to use cross-validation on the training data.

Best K using GridSearchCV

In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
("scaler", StandardScaler()),
("knn", KNeighborsClassifier())
])
param_grid = {
"knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
"knn__weights": ["uniform", "distance"],
"knn__metric": ["euclidean", "manhattan"]
}
grid = GridSearchCV(
pipeline,
param_grid,
cv=5,
scoring="f1",
n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)
best_knn = grid.best_estimator_

Best Parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
Best CV Score: 0.4426909589577823


Using a Pipeline ensures that scaling is fitted separately within each cross-validation training fold.

PART B – KNN REGRESSION

17. KNN Regression vs KNN Classification

KNN Classification KNN Regression
Predicts categories Predicts numbers
Uses majority voting Uses average or weighted average
Target may be Fraud/Genuine Target may be House Price
KNeighborsClassifier KNeighborsRegressor
Accuracy, Precision, Recall, F1 MAE, MSE, RMSE, R²

18. Complete KNN Regression Python Implementation

Step 1:  house-price dataset

In [3]:
import numpy as np
import pandas as pd
np.random.seed(42)
n = 1000
area = np.random.randint(500, 3000, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.randint(1, 40, n)
distance = np.random.uniform(1, 30, n)
price = (
0.04 * area
+ 8 * bedrooms
- 0.5 * age
- 1.2 * distance
+ np.random.normal(0, 5, n)
)
df = pd.DataFrame({
"Area": area,
"Bedrooms": bedrooms,
"Age": age,
"Distance": distance,
"Price": price
})
print(df.head())
print(df.shape)

   Area  Bedrooms  Age   Distance      Price
0  1360         1   22  16.700821  33.197262
1  1794         4    6   5.470206  83.695620
2  1630         5    2  17.057633  86.858380
3  1595         1   23  16.709440  44.674830
4  2138         1    2  23.095153  61.844037
(1000, 5)


In [4]:
df

,Area,Bedrooms,Age,Distance,Price
0,1360,1,22,16.700821,33.197262
1,1794,4,6,5.470206,83.695620
2,1630,5,2,17.057633,86.858380
3,1595,1,23,16.709440,44.674830
4,2138,1,2,23.095153,61.844037
...,...,...,...,...,...
995,1375,3,29,13.826520,48.105411
996,1262,4,13,21.514513,42.983037
997,1702,5,17,12.287684,78.255701
998,1705,2,32,7.637388,60.440180


This is synthetic house-price data. The formula is a simplified example designed to demonstrate regression.

Step 2: Separate X and y

In [12]:
X = df.drop("Price", axis=1)
y = df["Price"]

Step 3: Train-test split

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42
)

Step 4: Feature scaling

In [14]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Scaling is important because KNN uses distances.

Step 5: Create the model

In [15]:
from sklearn.neighbors import KNeighborsRegressor
knn_reg = KNeighborsRegressor(
n_neighbors=5,
weights="uniform",
metric="minkowski",
p=2
)

Step 6: Train

In [16]:
knn_reg.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"effective_metric_ effective_metric_: str or callableThe distance metric to use. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
"effective_metric_params_ effective_metric_params_: dictAdditional keyword arguments for the metric function. For most metricswill be same with `metric_params` parameter, but may also contain the`p` parameter value if the `effective_metric_` attribute is set to'minkowski'.",dict,{}


Step 7: Predict

In [17]:
y_pred = knn_reg.predict(X_test_scaled)

Step 8: Evaluate

In [18]:
from sklearn.metrics import (
mean_absolute_error,
mean_squared_error,
r2_score
)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(
mean_squared_error(y_test, y_pred)
))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 7.044694610560697
MSE: 73.64239011138851
RMSE: 8.58151444159995
R2 Score: 0.933133838913755


Step 9: Actual vs Predicted

In [19]:
comparison = pd.DataFrame({
"Actual_Price": y_test.values,
"Predicted_Price": y_pred
})
comparison["Difference"] = (
comparison["Actual_Price"]
- comparison["Predicted_Price"]
)
print(comparison.head(10))

   Actual_Price  Predicted_Price  Difference
0     25.561332        31.237975   -5.676643
1     63.603818        68.582731   -4.978913
2     28.841183        45.065631  -16.224448
3    137.017173       123.399737   13.617436
4     30.759735        20.394632   10.365104
5     46.982570        45.997991    0.984579
6     34.762427        43.678255   -8.915829
7     91.585208        91.168311    0.416897
8    140.231391       129.159267   11.072124
9    108.932847       108.289329    0.643518


19. KNN Hyperparameters

Parameter Purpose
n_neighbors Number of neighbors
weights Uniform or distance-based weighting
algorithm Neighbor-search algorithm
leaf_size Controls tree-based search efficiency
p Power parameter for Minkowski distance
metric Distance measurement
metric_params Additional distance metric parameters
n_jobs Number of parallel jobs

19.1 n_neighbors
Controls how many neighbors are selected.
KNeighborsClassifier(n_neighbors=5)

19.2 weights
Two important options:
Uniform:
KNeighborsClassifier(weights="uniform")
Every neighbor gets equal importance.
Distance:
KNeighborsClassifier(weights="distance")
Closer neighbors get more importance.

19.3 algorithm
Possible values:
 auto
 ball_tree
 kd_tree
 brute

KNeighborsClassifier(
n_neighbors=5,
algorithm="auto"
)

Explanation:
Brute Force
Calculates distances between the query observation and training observations directly.
KD Tree
Uses a tree-based structure to organize observations and accelerate certain neighbor searches.
Ball Tree
Uses a tree structure based on distances from center points to organize observations.
Auto
Scikit-learn selects an appropriate search method based on the data and configuration.
The best algorithm depends on the dataset’s dimensions, size, and metric.

19.4 p
Used with Minkowski distance.
 p = 1: Manhattan distance.
 p = 2: Euclidean distance.
Example:
KNeighborsClassifier(
n_neighbors=5,
metric="minkowski",
p=1
)
This uses Manhattan distance.

19.5 n_jobs
Controls parallel processing for supported neighbor-search operations.
KNeighborsClassifier(
n_neighbors=5,
n_jobs=-1
)
n_jobs=-1 uses all available CPU cores supported by the operation.

20. What is the Curse of Dimensionality?
This is an important concept in KNN.
KNN depends on distance calculations.
When the number of features becomes very large, the concept of nearest and farthest observations can become less meaningful.
This problem is called the Curse of Dimensionality.

Example
Imagine finding the closest person in a room.
If you compare people using only two characteristics:
 Height
 Age
It may be relatively easy to identify similar people.
Now imagine comparing them using 1,000 characteristics.
The space becomes much more complex, and observations can become sparse.
Distance differences may become less informative.
This affects KNN because its predictions depend on meaningful neighborhoods.

How to handle it?
1. Remove irrelevant features.
2. Perform feature selection.
3. Use PCA when appropriate.
4. Remove redundant variables.
5. Apply domain-based feature engineering.
6. Evaluate different distance metrics.
7. Use cross-validation.

PCA with KNN
PCA reduces the number of dimensions while attempting to retain as much variance as possible.
Example:

In [20]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print("Original Features:", X_train_scaled.shape[1])
print("PCA Features:", X_train_pca.shape[1])

Original Features: 4
PCA Features: 4


Here, 95% means PCA selects enough components to retain approximately 95% of the training data’s variance.
After PCA, we can train KNN using the transformed features.
PCA must be fitted only on training data. In cross-validation, put PCA inside a Pipeline to prevent leakage.